## byLLM Analysis

This is a byLLM analysis notebook analyzing the differences between the ground-truth dataset and the cleaned (byLLM) dataset. 

byLLM dataset uses Google Gemini's 2.5 Flash model via byLLM. Refer to `/cleaning/iowa_cleaning/byLLM_cleaning.ipynb` for more details.

#### Analysis

In [1]:
# import packages
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [2]:
# import data (only 250 rows for now)
df_manual1 = pd.read_csv("../../data/iowa_liquor/mt_cleaned_iowa_liquor_250.csv")
df_llm = pd.read_csv("../../data/iowa_liquor/byllm_cleaned_iowa_liquor_250.csv")
df_manual = df_manual1.iloc[:250, :5]

In [3]:
# schema integrity
manual_cols = set(df_manual.columns)
llm_cols = set(df_llm.columns)

schema_diff = {
    "dropped_columns": list(manual_cols - llm_cols),
    "invented_columns": list(llm_cols - manual_cols),
    "row_count_diff": len(df_llm) - len(df_manual)
}

print("schema differences:")
print("-"*20)
print(schema_diff)

schema differences:
--------------------
{'dropped_columns': [], 'invented_columns': [], 'row_count_diff': 0}


In [4]:
# make sure both csvs have same "primary key" due to comparisions
df_manual = df_manual.set_index("Invoice/Item Number")
df_llm = df_llm.set_index("Invoice/Item Number")

commons = df_manual.index.intersection(df_llm.index)

df_manual = df_manual.loc[commons]
df_llm = df_llm.loc[commons]

In [5]:
# calculate metrics for all cols
precisions = []
recalls = []
f1s = []

for col in df_llm.columns:
    
    if col not in df_manual.columns:
        continue
    
    y_true = df_manual[col].fillna("MISSING").astype(str)
    y_pred = df_llm[col].fillna("MISSING").astype(str)
    
    precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    
    precisions.append(precision)
    recalls.append(recall)
    f1s.append(f1)

In [6]:
# find avgs for all cols
avg_precision = np.mean(precisions)
avg_recall = np.mean(recalls)
avg_f1 = np.mean(f1s)

print("\navg metrics for all cols:")
print("-"*25)
print(f"avg precision: {avg_precision:.4f}")
print(f"avg recall:    {avg_recall:.4f}")
print(f"avg F1 score:  {avg_f1:.4f}")


avg metrics for all cols:
-------------------------
avg precision: 0.8200
avg recall:    0.8200
avg F1 score:  0.8200


In [7]:
# misinterpretation rate: the proportion of records with 1+ violation/incorrect repairs across all columns

common_cols = [col for col in df_llm.columns if col in df_manual.columns]

# a row is "misinterpreted" if ANY column value differs from ground truth
mismatches = (df_manual[common_cols].fillna("MISSING").astype(str) !=
              df_llm[common_cols].fillna("MISSING").astype(str))

misinterpreted_rows = mismatches.any(axis=1).sum()
total_rows = len(df_manual)
misinterpretation_rate = misinterpreted_rows / total_rows

print("misinterpretations:")
print("-"*20)
print(f"rows: {misinterpreted_rows} / {total_rows}")
print(f"rate: {misinterpretation_rate:.4f} ({misinterpretation_rate*100:.2f}%)")

misinterpretations:
--------------------
rows: 161 / 250
rate: 0.6440 (64.40%)


In [8]:
# column-level mismatches
col_mismatch_rates = mismatches.mean().sort_values(ascending=False)

In [9]:
print("top 15 most mismatched columns:")
print("-"*35)
print(col_mismatch_rates.head(15).to_string())

top 15 most mismatched columns:
-----------------------------------
Store Name      0.548
Address         0.172
Date            0.000
Store Number    0.000


In [10]:
print("bottom 15 least mismatched columns (most accurate):")
print("-"*51)
print(col_mismatch_rates.tail(15).to_string())

bottom 15 least mismatched columns (most accurate):
---------------------------------------------------
Store Name      0.548
Address         0.172
Date            0.000
Store Number    0.000


In [11]:
print("none or total mismatches:")
print("-"*25)
print(f"columns with 0% mismatch: {(col_mismatch_rates == 0).sum()}")
print(f"columns with 100% mismatch: {(col_mismatch_rates == 1).sum()}")

none or total mismatches:
-------------------------
columns with 0% mismatch: 2
columns with 100% mismatch: 0


In [12]:
# summarize mismatches by columns
print(f"total shared columns: {len(common_cols)}")
print(f"columns with 0% mismatch: {(col_mismatch_rates == 0).sum()} ({(col_mismatch_rates == 0).mean()*100:.1f}%)")
print(f"columns with <5% mismatch: {(col_mismatch_rates < 0.05).sum()} ({(col_mismatch_rates < 0.05).mean()*100:.1f}%)")
print(f"columns with 5-25% mismatch: {((col_mismatch_rates >= 0.05) & (col_mismatch_rates < 0.25)).sum()}")
print(f"columns with >25% mismatch: {(col_mismatch_rates >= 0.25).sum()}")

total shared columns: 4
columns with 0% mismatch: 2 (50.0%)
columns with <5% mismatch: 2 (50.0%)
columns with 5-25% mismatch: 1
columns with >25% mismatch: 1
